# Offline policy learning (stochastic, discrete H=2, M=2)

We use a tiny discrete MDP with 3 states and 3 actions. Rewards are two-dimensional and conflicting (positive actions help objective 1 while hurting objective 2; negative actions do the opposite). The learned policy is stochastic with an epsilon floor so every action keeps some probability mass for exploration.

In [2]:
import numpy as np
from utils import (
    FiniteHorizonMultiObjectiveMDP,
    evaluate_policy_value,
    generate_dataset,
    learn_policy_from_dataset,
)

np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Build the MDP and generate offline data
mdp = FiniteHorizonMultiObjectiveMDP(seed=7)
dataset = generate_dataset(mdp, num_trajectories=4000, seed=123, behavior_min_prob=0.05)
len(dataset)

4000

In [4]:
states = [-1, 0, 1]
actions = [-1, 0, 1]

def format_probs(probs):
    return ", ".join(f"{a:+d}:{p:.2f}" for a, p in zip(actions, probs))

def print_policy(label, w, policy_info, value_estimate):
    print(f"Preference w={label}, estimated V={value_estimate:.3f}")
    for si, s in enumerate(states):
        p1 = policy_info["stage1_probs"][si]
        p2 = policy_info["stage2_probs"][si]
        print(f"  s={s:+d} stage1: a~[{format_probs(p1)}]")
        print(f"  s={s:+d} stage2: a~[{format_probs(p2)}]")
    print("")

In [5]:
# Train policies for different preferences and report their action distributions/value
weights = {
    "(1, 0)": np.array([1.0, 0.0]),
    "(0, 1)": np.array([0.0, 1.0]),
    "(0.5, 0.5)": np.array([0.5, 0.5]),
}

for label, w in weights.items():
    policy, info = learn_policy_from_dataset(dataset, w, min_prob=0.05, temperature=0.1)
    value_estimate = evaluate_policy_value(mdp, policy, w, num_rollouts=3000)
    print_policy(label, w, info, value_estimate)

Preference w=(1, 0), estimated V=3.233
  s=-1 stage1: a~[-1:0.02, +0:0.02, +1:0.96]
  s=-1 stage2: a~[-1:0.02, +0:0.06, +1:0.92]
  s=+0 stage1: a~[-1:0.02, +0:0.02, +1:0.96]
  s=+0 stage2: a~[-1:0.02, +0:0.06, +1:0.92]
  s=+1 stage1: a~[-1:0.02, +0:0.06, +1:0.92]
  s=+1 stage2: a~[-1:0.02, +0:0.06, +1:0.92]

Preference w=(0, 1), estimated V=3.240
  s=-1 stage1: a~[-1:0.92, +0:0.06, +1:0.02]
  s=-1 stage2: a~[-1:0.92, +0:0.06, +1:0.02]
  s=+0 stage1: a~[-1:0.96, +0:0.02, +1:0.02]
  s=+0 stage2: a~[-1:0.92, +0:0.06, +1:0.02]
  s=+1 stage1: a~[-1:0.96, +0:0.02, +1:0.02]
  s=+1 stage2: a~[-1:0.92, +0:0.06, +1:0.02]

Preference w=(0.5, 0.5), estimated V=2.539
  s=-1 stage1: a~[-1:0.04, +0:0.91, +1:0.04]
  s=-1 stage2: a~[-1:0.04, +0:0.91, +1:0.04]
  s=+0 stage1: a~[-1:0.04, +0:0.91, +1:0.04]
  s=+0 stage2: a~[-1:0.04, +0:0.91, +1:0.04]
  s=+1 stage1: a~[-1:0.04, +0:0.91, +1:0.04]
  s=+1 stage2: a~[-1:0.04, +0:0.91, +1:0.04]

